In [0]:
-- Все проверки должны быть = 0 

SELECT
  -- 1. уникальность
  COUNT(*) - COUNT(DISTINCT order_id)                       AS duplicate_orders,   -- дубликаты заказов
  -- 2. обязательные поля не пустые (NULL после cast = значение не распарсилось)
  COUNT_IF(order_id IS NULL)                                AS null_order_id,
  COUNT_IF(order_ts IS NULL)                                AS null_order_ts,      -- дата не распарсилась
  COUNT_IF(customer_id IS NULL)                             AS null_customer,
  COUNT_IF(quantity IS NULL OR unit_price IS NULL)          AS null_qty_or_price,
  COUNT_IF(currency IS NULL)                                AS null_currency,
  -- 3. бизнес-правила
  COUNT_IF(quantity <= 0)                                   AS qty_le_0,
  COUNT_IF(unit_price <= 0)                                 AS price_le_0,
  COUNT_IF(order_ts > current_timestamp())                  AS future_orders,      -- заказы из будущего
  -- 4. допустимые значения (важно для gold: курс валюты должен найтись)
  COUNT_IF(currency NOT IN ('CZK','EUR','USD','GBP','PLN')) AS unknown_currency,
  COUNT_IF(status NOT IN ('paid','cancelled','refunded'))   AS unknown_status
FROM workspace.shop.silver_orders;


In [0]:
-- Цель: сверка bronze → silver: сколько строк выброшено и почему
SELECT
  (SELECT COUNT(*) FROM workspace.shop.bronze_orders) AS bronze_rows,   -- пришло
  (SELECT COUNT(*) FROM workspace.shop.silver_orders) AS silver_rows,   -- осталось
  (SELECT COUNT(*) FROM workspace.shop.bronze_orders)
  - (SELECT COUNT(*) FROM workspace.shop.silver_orders) AS dropped_rows; -- выброшено

In [0]:
-- проверка: 5 валют на каждый день
SELECT * FROM workspace.shop.silver_rates
WHERE rate_date BETWEEN '2026-10-20' AND '2026-10-22'
ORDER BY rate_date, currency;   -- ожидаем 15 строк; CZK ≈ 0.04, USD ≈ 0.9, EUR = 1